In [2]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
   전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}

</style>
"""))


# <span style="color:red">ch3. 연관분석<span>
- pip install apyori

# 1. 연관분석 개요
- 데이터들 사이의 자주 발생하는 속성을 찾고, 그 속성들 사이의 연관성이 어느 정도있는지를 분석
- 활용분야 : 상품진열, 사기보험적발, 신상품 카테고리 구성, ...
```
조건(left-hand side, item_base) : 오렌지주스(x) => 결과(right-hand side, item_add) : 와인(y)

연관분석 지표
1. 지지도(support) : 전체 데이터 중, 조건과 결과 항목들이 포함된 거래 비율(함께 얼마나 자주 나타나는지)
        (x, y)의 항목수 / 전체 데이터수 = 0.2
2. 신뢰도(confidence) : 조건(x)이 발생했을 때, 결과가 동시에 일어날 확률(조건이 오면 얼마나 자주 결과가 오는지)
      (x=>y)의 항목수 / x가 나오는 항목수 = 0.5
3. 향상도(lift) : 우연히 발생할 규칙은 아니였는지 확인
       1미만 : 독립적으로 나오는 것보다 함께 나타날 가능성이 낮다
       1       : 서로 독립적. 아무 연관성 없다
       1초과 : 양의 상관관계(같이 잘 나온다)
       (x=>y)의 지지도 / (x의 지지도*y의 지지도) = 0.2 / (0.4*0.6) = 0.2/0.24 = 0.83333...
```
# 2. 연관분석 구현

In [3]:
import csv
with open('data/cf_basket.csv', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [11]:
from apyori import apriori
rules = apriori(
    transaction, 
    min_support=0.15, 
    min_confidence=0.1,
    min_lift=1.001 )
rules = list(rules)
len(rules)

6

In [12]:
rule = rules[5]
rule

RelationRecord(items=frozenset({'와인', '소주', '콜라'}), support=0.2, ordered_statistics=[OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'와인', '소주'}), confidence=0.25, lift=1.25), OrderedStatistic(items_base=frozenset({'와인', '소주'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25)])

In [23]:
rules_lst = [] # 규칙을 저장할 dict list
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
       # print(item)
        lhs = item[0]
        lhs = ','.join([x for x in lhs])
        rhs = item[1]
        rhs = ','.join([x for x in rhs])
        confidence = item[2]
        lift = item[3]
#    print(f"{lhs}=>{rhs} \t {support} \t {round(confidence,2)} \t {round(lift,2)}")
#        rules_lst.append({'lhs':lhs, 'rhs':rhs, 'support':support, 'confidence':round(confidence,2), 'lift':round(lift,2)})
        rules_lst.append([lhs, rhs, support, round(confidence, 2), round(lift, 2)])
import pandas as pd
pd.DataFrame(rules_lst, columns=['lhs', 'rhs', 'support', 'confidence', 'lift'])

,lhs,rhs,support,confidence,lift
0,맥주,콜라,0.4,1.00,1.25
1,콜라,맥주,0.4,0.50,1.25
2,소주,콜라,0.6,1.00,1.25
3,콜라,소주,0.6,0.75,1.25
4,콜라,"맥주,소주",0.2,0.25,1.25
5,"소주,맥주",콜라,0.2,1.00,1.25
6,맥주,"와인,콜라",0.2,0.50,1.25
7,콜라,"와인,맥주",0.2,0.25,1.25
8,"와인,맥주",콜라,0.2,1.00,1.25
9,"와인,콜라",맥주,0.2,0.50,1.25


# 3. 뉴스 연관분석
- 경제뉴스 20개를 각각 명사만 가져와서 list (naver API) -> 연관분석
    ```
    [['단어1', '단어2', '단어3',...],
     ['단어4', 단어5', '단어6', ...],
     ['단어2', '단어7', '단어1',...]...]
    ```

In [24]:
from dotenv import load_dotenv
import os
load_dotenv() # .env의 시스템 환경변수 불러오기

True

In [37]:
# 방법 1 예시
import json
import os
import sys
import requests
import pandas as pd
from dotenv import load_dotenv
from requests import get
load_dotenv() # 환경 변수 load
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

url = f"https://openapi.naver.com/v1/search/news.json" # JSON 결과
params = {'query': '경제', 'display':20, 'sort':'date' } # 'display' : 20  = 가져올 데이터 개수  / 'sort':'date' = 최신순 뉴스
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items']# response를 json형태로 변환한것 중 'items' /  방법 1만 가능

# 제목(title) + ' ' + 요약(description) 텍스만 추출하여 list
news_texts = []
for item in items:
    title = item.get('title').replace('<b>', ' ').replace('</b>', ' ')
    description = item.get('description').replace('<b>', ' ').replace('</b>', ' ')
    news_texts.append(title + ' ' + description)
news_texts[:3], len(news_texts)

(["용담2동주민자치회 '소비 심백(心 百)캠페인' 전개 후 골목상권 활성화를 위한 '소비 심백 캠패인'을 전개했다. 이날 참여한 회원들은 골목상권 이용 및 지역상가와 전통시장을 방문해 주민들의 적극적인 참여를 독려하고 지역 경제 활성화에 적극 노력하기로 했다.",
  "천안시의회  경제 산업위, 'AI 중심 도시' 도약 위한 비교견학 ［중부매일 송문용 기자］ 천안시의회  경제 산업위원회(위원장 복아영)는 인공지능(AI) 기술이 전 산업의...  경제 산업위원회는 의원 7명과 사무국 직원 4명 등 총 11명으로 방문단을 구성해, 7일부터 8일까지 1박2일 일정으로... ",
  '제주 풍수해·지진재해보험료 경감...도, 지원 확대 주민과  경제 적 취약계층 등의 보험 가입을 확대하기 위해 지원을 늘렸다. 풍수해·지진재해보험은 태풍... 일부  경제 적 취약계층에 대한 지원율을 높인 것이다. 일반 세입자는 기존 지방비 추가지원 대상에서 제외됐으나... '],
 20)

In [38]:
# 명사추출(news_texts)
stopwords = {'기사', '기자'}
from konlpy.tag import Hannanum, Kkma, Komoran, Okt
from mecab import MeCab
analyzer = MeCab()
news = []
for article in news_texts:
    noun_list = analyzer.nouns(article)
    noun_list = [word for word, tag in analyzer.pos(article) \
                 if tag in ('NNG','NNP') and # 명사
                    word not in stopwords and # 불용어(stopwords) 제외
                    len(word)>1 ] # 2글자 이상의 단어
    news.append(noun_list)
print(news[:3])

[['용담', '주민', '자치회', '소비', '캠페인', '전개', '목상', '활성', '소비', '패인', '전개', '이날', '참여', '회원', '목상', '이용', '지역', '상가', '전통', '시장', '방문', '주민', '적극', '참여', '독려', '지역', '경제', '활성', '적극', '노력'], ['천안시', '의회', '경제', '산업', '중심', '도시', '도약', '비교', '견학', '중부', '송문', '천안시', '의회', '경제', '산업', '위원회', '위원장', '아영', '인공지능', '기술', '산업', '경제', '산업', '위원회', '의원', '사무국', '직원', '방문단', '구성', '일정'], ['제주', '풍수', '지진', '재해', '보험료', '경감', '지원', '확대', '주민', '경제', '취약', '계층', '보험', '가입', '확대', '지원', '풍수', '지진', '재해', '보험', '태풍', '일부', '경제', '취약', '계층', '지원', '일반', '세입자', '기존', '지방비', '추가', '지원', '대상', '제외']]


In [42]:
rules = apriori(news,
               min_support=0.15,
               min_confidence=0.1,
               min_lift=1.0001)
rules = list(rules)
len(rules)

12

In [43]:
rules_lst = [] # 규칙을 저장할 dict list
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
       # print(item)
        lhs = item[0]
        lhs = ','.join([x for x in lhs])
        rhs = item[1]
        rhs = ','.join([x for x in rhs])
        confidence = item[2]
        lift = item[3]
#    print(f"{lhs}=>{rhs} \t {support} \t {round(confidence,2)} \t {round(lift,2)}")
#        rules_lst.append({'lhs':lhs, 'rhs':rhs, 'support':support, 'confidence':round(confidence,2), 'lift':round(lift,2)})
        rules_lst.append([lhs, rhs, support, round(confidence, 2), round(lift, 2)])
import pandas as pd
df = pd.DataFrame(rules_lst, columns=['lhs', 'rhs', 'support', 'confidence', 'lift'])

In [52]:
df.sort_values(by=['lift','confidence', 'support'], ascending=False, inplace=True)
df # 지지도는 신뢰성 체크용

,lhs,rhs,support,confidence,lift
34,"사업,지역",이번,0.15,1.00,4.00
46,"사업,경제,지역",이번,0.15,1.00,4.00
43,"사업,지역","이번,경제",0.15,1.00,4.00
31,이번,"사업,지역",0.15,0.60,4.00
37,이번,"사업,경제,지역",0.15,0.60,4.00
40,"이번,경제","사업,지역",0.15,0.60,4.00
47,"이번,경제,지역",사업,0.15,0.75,3.75
30,사업,"이번,지역",0.15,0.75,3.75
35,"이번,지역",사업,0.15,0.75,3.75
44,"이번,지역","사업,경제",0.15,0.75,3.75
